In [2]:
!pip install pycuda
!apt-get install nvidia-cuda-toolkit

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.8/92.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.4 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2025.1-cp311-cp311-linux_x86_64.whl size=660394 sha256=c4db6101053dfb353ab42f970d1dd691d48bffaa7fe03102cfa6d67d67b2d490
  Stored in directory: /root/.cache/pip/wheels/77/7e/6c/d2d1451ea6424cdc3d67b36c16fa7111eafdf2034bc3405666
Successfully built pycuda
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libaccinj64-11.5 libatk-wrapper-java libatk-wrapper-java-jni
  libbabeltrace1 libcub-dev libcublas11 libcublaslt11 libcudart11.0 libcufft10 libc

In [ ]:
TAM_IN = 200
TAM_EX = 20000

def random_entrada(tamanho_interno=200) :
  idx = blockIdx.x * blockDim.x + threadIdx.x
  inputvector = np.random.randint(2, size=(tamanho_interno), dtype=np.int32)
  return inputvector

def criar_vetores(tamanho_externo=20000, tamanho_interno=200):
    """
    Cria linevector e valuevector separadamente.
    linevector: vetor de vetor de binários com 20% de zeros
    valuevector: vetor de vetor de binários aleatórios
    """
    # Inicializa o linevector com uns
    linevector = np.ones((tamanho_externo, tamanho_interno), dtype=np.int32)
    #sorteia valores(0 ou 1) para o valuevector
    valuevector = np.random.randint(2, size=(tamanho_externo, tamanho_interno), dtype=np.int32)

    # Calcula quantos zeros devem ter em cada vetor interno do linevector (20%)
    num_zeros = int(tamanho_interno * 0.2)

    # Para cada vetor interno do linevector
    for i in range(tamanho_externo):
        # Escolhe aleatoriamente as posições que serão zero
        posicoes_zero = np.random.choice(tamanho_interno, size=num_zeros, replace=False)
        # Define os zeros nas posições escolhidas
        linevector[i, posicoes_zero] = 0

    return linevector, valuevector

# instanciando para passar para o kernel 2
linevector, valuevector = criar_vetores()

kernel_code = """
__global__ void simulate_kernel(int *linevector, int *valuevector, int *inputvector,
                              int *outputvector, int N_input, int tamInputvector)
{
    // Cálculo do idx usando idblk (blockIdx.x) e idth (threadIdx.x)
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int output_sum = 0;  // Acumulador para somar todos os resultados

    // Loop principal sobre o inputvector
    for (int i = 0; i < tamInputvector; i++) {
        // Inicialização dos registradores de entrada
        int a0 = inputvector[0 + i * N_input];
        int a1 = inputvector[1 + i * N_input];
        int a2 = inputvector[2 + i * N_input];
        int a3 = inputvector[3 + i * N_input];

        // Avaliação do prune para cada bit
        int prune_output_word = 0;

        new = np.array()
        for i in range TAM_IN
          new[i] = (linevector[idx,i] == 0) ? valuevector[idx,i] : equaçao lut
          prune_output_world |= ((new[i] & 1) << i);

        // Calcula XOR e conta bits diferentes usando __popc
        output_sum += __popc(true_output_word ^ prune_output_word);
    }

    // Armazena o resultado final no outputvector
    outputvector[idx] = output_sum;
}
"""

def simulate(linevector, valuevector, inputvector, N_input=4, num_blocks=256, threads_per_block=256):
    """
    Função principal para executar a simulação
    """
    # Compilação do módulo CUDA
    mod = SourceModule(kernel_code)
    simulate_kernel = mod.get_function("simulate_kernel")

    # Tamanho do vetor de entrada (2^20)
    tamInputvector = 2**20

    # Preparação dos vetores para CUDA
    linevector = linevector.astype(np.int32)
    valuevector = valuevector.astype(np.int32)
    inputvector = inputvector.astype(np.int32)

    # Vetor de saída
    output_size = num_blocks * threads_per_block
    outputvector = np.zeros(output_size, dtype=np.int32)

    # Execução do kernel
    simulate_kernel(
        cuda.In(linevector),
        cuda.In(valuevector),
        cuda.In(inputvector),
        cuda.Out(outputvector),
        np.int32(N_input),
        np.int32(tamInputvector),
        block=(threads_per_block, 1, 1),
        grid=(num_blocks, 1)
    )

    return outputvector


SyntaxError: invalid syntax (<ipython-input-1-a790fdf5070c>, line 6)

In [1]:
# Kernel CUDA para inicialização e geração de vetores
kernel_code = """
#include <curand_kernel.h>

__global__ void criar_vetores(int *linevector, int *valuevector, int tamanho_externo, int tamanho_interno, int num_zeros) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < tamanho_externo) {
        // Inicializa todos os elementos como 1
        for (int i = 0; i < tamanho_interno; i++) {
            linevector[idx * tamanho_interno + i] = 1;
            valuevector[idx * tamanho_interno + i] = (rand() % 2);  // 0 ou 1
        }

        // Define uma seed para cada thread para garantir aleatoriedade diferente
        curandState state;
        curand_init(clock64(), idx, 0, &state);

        // Define 20% dos valores como zero em linevector
        for (int j = 0; j < num_zeros; j++) {
            int pos = curand(&state) % tamanho_interno;
            linevector[idx * tamanho_interno + pos] = 0;
        }
    }
}
"""

# Compilar o kernel CUDA
mod = SourceModule(kernel_code)
criar_vetores_kernel = mod.get_function("criar_vetores")

# Alocar memória no host (CPU)
linevector_host = np.ones((TAMANHO_EXTERNO, TAMANHO_INTERNO), dtype=np.int32)
valuevector_host = np.random.randint(2, size=(TAMANHO_EXTERNO, TAMANHO_INTERNO), dtype=np.int32)

# Alocar memória na GPU
linevector_gpu = cuda.mem_alloc(linevector_host.nbytes)
valuevector_gpu = cuda.mem_alloc(valuevector_host.nbytes)

# Copiar dados iniciais para GPU
cuda.memcpy_htod(linevector_gpu, linevector_host)
cuda.memcpy_htod(valuevector_gpu, valuevector_host)

# Configuração do grid e blocos
threads_por_bloco = 256
blocos_por_grid = (TAMANHO_EXTERNO + threads_por_bloco - 1) // threads_por_bloco

# Executar o kernel CUDA
criar_vetores_kernel(
    linevector_gpu,
    valuevector_gpu,
    np.int32(TAMANHO_EXTERNO),
    np.int32(TAMANHO_INTERNO),
    np.int32(int(TAMANHO_INTERNO * PORCENTAGEM_ZEROS)),
    block=(threads_por_bloco, 1, 1),
    grid=(blocos_por_grid, 1, 1)
)

# Copiar os resultados de volta para a CPU
cuda.memcpy_dtoh(linevector_host, linevector_gpu)
cuda.memcpy_dtoh(valuevector_host, valuevector_gpu)

# Exibir uma parte dos dados gerados
print("Exemplo de linevector:")
print(linevector_host[:5])  # Mostra as primeiras 5 linhas
print("Exemplo de valuevector:")
print(valuevector_host[:5])

NameError: name 'SourceModule' is not defined

In [3]:
import numpy as np
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import pycuda.gpuarray as gpuarray

# Kernel CUDA em C
cuda_code = """
#include <curand_kernel.h>

extern "C" {
    __global__ void create_vectors(int *linevector, int *valuevector,
                                 int tamanho_interno, unsigned long long seed) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        int offset = idx * tamanho_interno;

        // Inicializa o gerador de números aleatórios
        curandState state;
        curand_init(seed, idx, 0, &state);

        // Preenche o subvetor do linevector com 1s
        for(int i = 0; i < tamanho_interno; i++) {
            linevector[offset + i] = 1;
        }

        // Escolhe 20% das posições para serem zero no linevector
        int num_zeros = (int)(tamanho_interno * 0.2);
        for(int i = 0; i < num_zeros; i++) {
            int pos;
            do {
                pos = (int)(curand_uniform(&state) * tamanho_interno);
            } while(linevector[offset + pos] == 0);  // Evita escolher a mesma posição

            linevector[offset + pos] = 0;
        }

        // Preenche o subvetor do valuevector com valores aleatórios (0 ou 1)
        for(int i = 0; i < tamanho_interno; i++) {
            valuevector[offset + i] = (curand_uniform(&state) > 0.5) ? 1 : 0;
        }
    }
}
"""

def criar_vetores_gpu(tamanho_externo=20000, tamanho_interno=200):
    """
    Cria linevector e valuevector em paralelo na GPU.
    linevector: vetor de vetor de binários com 20% de zeros
    valuevector: vetor de vetor de binários aleatórios
    """
    # Compila o módulo CUDA
    mod = SourceModule(cuda_code)

    # Obtém a função do kernel
    create_vectors = mod.get_function("create_vectors")

    # Aloca memória na GPU
    linevector_gpu = cuda.mem_alloc(tamanho_externo * tamanho_interno * np.int32().nbytes)
    valuevector_gpu = cuda.mem_alloc(tamanho_externo * tamanho_interno * np.int32().nbytes)

    # Configura a grade de threads
    block_size = 256
    grid_size = (tamanho_externo + block_size - 1) // block_size

    # Executa o kernel
    create_vectors(
        linevector_gpu,
        valuevector_gpu,
        np.int32(tamanho_interno),
        np.uint64(np.random.randint(2**32)),  # seed aleatória
        block=(block_size, 1, 1),
        grid=(grid_size, 1)
    )

    # Aloca arrays no host para receber os resultados
    linevector = np.empty((tamanho_externo, tamanho_interno), dtype=np.int32)
    valuevector = np.empty((tamanho_externo, tamanho_interno), dtype=np.int32)

    # Copia os resultados de volta para o host
    cuda.memcpy_dtoh(linevector, linevector_gpu)
    cuda.memcpy_dtoh(valuevector, valuevector_gpu)

    return linevector, valuevector

# Exemplo de uso
if __name__ == "__main__":
    # Para executar no Google Colab, primeiro instale o PyCUDA:
    # !pip install pycuda

    linevector, valuevector = criar_vetores_gpu()

    # Verificação dos resultados
    print("Formato do linevector:", linevector.shape)
    print("Formato do valuevector:", valuevector.shape)

    # Verifica se cada linha do linevector tem 20% de zeros
    zeros_por_linha = np.sum(linevector == 0, axis=1)
    print("Média de zeros por linha:", zeros_por_linha.mean())
    print("Deve ser próximo a:", 200 * 0.2)


CompileError: nvcc compilation of /tmp/tmpyl_bhz9d/kernel.cu failed
[command: nvcc --cubin -arch sm_75 -I/usr/local/lib/python3.11/dist-packages/pycuda/cuda kernel.cu]
[stderr:
/usr/local/cuda/bin/../targets/x86_64-linux/include/curand_kernel.h(315): error: this declaration may not have extern "C" linkage
  template<int N>
  ^

/usr/local/cuda/bin/../targets/x86_64-linux/include/curand_kernel.h(569): error: this declaration may not have extern "C" linkage
  template <typename T, int n>
  ^

/usr/local/cuda/bin/../targets/x86_64-linux/include/curand_kernel.h(589): error: this declaration may not have extern "C" linkage
  template <typename T, int n>
  ^

/usr/local/cuda/bin/../targets/x86_64-linux/include/curand_kernel.h(645): error: this declaration may not have extern "C" linkage
  template <typename T, int n>
  ^

/usr/local/cuda/bin/../targets/x86_64-linux/include/curand_kernel.h(701): error: this declaration may not have extern "C" linkage
  template <typename T, int N>
  ^

/usr/l